# 03 Student Baseline (CE)

Cel: wytrenowac baseline model student tylko na etykietach (CrossEntropy) z jawna petla treningowa.


In [1]:
# Krok 1: importy i konfiguracja
# Po co to robimy: przygotowujemy wszystkie biblioteki i stale, aby trening baseline byl powtarzalny i czytelny.

# Importujemy json, aby zapisac metryki do pliku.
# Importujemy modul os do konfiguracji cache i warningow HF.
import os
# Ustawiamy flage, aby ukryc warning o symlinkach na Windows.
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
# Ustawiamy lokalny cache HF w repo, aby latwiej panowac nad plikami.
os.environ.setdefault("HF_HOME", "../artifacts/hf_cache")
# Wylaczamy Xet backend, aby nie dostawac komunikatu o hf_xet.
os.environ["HF_HUB_DISABLE_XET"] = "1"

import json
# Importujemy PyTorch, bo na nim bedzie caly trening modelu.
import torch
# Importujemy loader danych z dysku, bo tokenizacja byla zapisana w kroku 02.
from datasets import load_from_disk
# Importujemy metryki klasyfikacji do oceny modelu.
from sklearn.metrics import accuracy_score, f1_score
# Importujemy funkcje do clippingu gradientow, aby stabilizowac trening.
from torch.nn.utils import clip_grad_norm_
# Importujemy DataLoader do wygodnego batchowania danych.
from torch.utils.data import DataLoader
# Importujemy pasek postepu, aby widziec przebieg treningu.
from tqdm.auto import tqdm
# Importujemy model klasyfikacyjny i tokenizer z biblioteki Transformers.
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Ustalamy nazwe modelu studenta, ktory bedziemy fine-tunowac.
STUDENT_MODEL_NAME = "prajjwal1/bert-tiny"
# Ustalamy rozmiar batcha treningowego.
BATCH_SIZE = 8
# Ustalamy liczbe epok treningu baseline.
EPOCHS = 4
# Ustalamy learning rate dla optymalizatora AdamW.
LR = 2e-5
# Ustalamy maksymalna norme gradientu dla clippingu.
MAX_GRAD_NORM = 1.0
# Wymuszamy trening na CPU, bo projekt jest CPU-friendly.
DEVICE = torch.device("cpu")


In [2]:
# Krok 2: ladowanie danych
# Po co to robimy: wczytujemy gotowe tokeny i budujemy DataLoadery, aby model dostawal dane w batchach.

# Wczytujemy z dysku tokenized dataset przygotowany w notebooku 02.
student_ds = load_from_disk("../artifacts/tokenized_student")
# Tworzymy loader treningowy z tasowaniem danych.
train_loader = DataLoader(student_ds["train"], batch_size=BATCH_SIZE, shuffle=True)
# Tworzymy loader walidacyjny bez tasowania.
val_loader = DataLoader(student_ds["validation"], batch_size=BATCH_SIZE, shuffle=False)
# Tworzymy loader testowy bez tasowania.
test_loader = DataLoader(student_ds["test"], batch_size=BATCH_SIZE, shuffle=False)

# Krok 3: budowa modelu i optymalizatora
# Po co to robimy: tworzymy model, funkcje straty i optymalizator, czyli komplet do uczenia baseline CE.

# Wczytujemy model klasyfikacyjny z 3 klasami sentymentu.
model = AutoModelForSequenceClassification.from_pretrained(STUDENT_MODEL_NAME, num_labels=3)
# Przenosimy model na wybrane urzadzenie (CPU).
model.to(DEVICE)
# Tworzymy optymalizator AdamW na parametrach modelu.
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
# Definiujemy klasyczna funkcje straty CrossEntropy dla klasyfikacji wieloklasowej.
criterion = torch.nn.CrossEntropyLoss()


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
# Krok 4: funkcja ewaluacji
# Po co to robimy: osobno mierzymy jakosc modelu na validation/test bez uczenia i bez gradientow.

# Definiujemy funkcje oceny modelu na dowolnym DataLoaderze.
def evaluate(model, dataloader):
    # Przelaczamy model w tryb ewaluacji (np. dropout off).
    model.eval()
    # Tworzymy liste na predykcje modelu.
    preds = []
    # Tworzymy liste na prawdziwe etykiety.
    labels = []
    # Wylaczamy gradienty, bo tylko mierzymy i to oszczedza pamiec/czas.
    with torch.no_grad():
        # Iterujemy po kolejnych batchach danych.
        for batch in dataloader:
            # Pobieramy input_ids i przenosimy na CPU.
            input_ids = batch["input_ids"].to(DEVICE)
            # Pobieramy attention_mask i przenosimy na CPU.
            attention_mask = batch["attention_mask"].to(DEVICE)
            # Pobieramy prawdziwe etykiety i przenosimy na CPU.
            y = batch["label"].to(DEVICE)
            # Robimy forward modelu i pobieramy logits.
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            # Zamieniamy logits na klasy przez argmax i zapisujemy je do listy predykcji.
            preds.extend(torch.argmax(logits, dim=-1).cpu().tolist())
            # Dodajemy prawdziwe etykiety do listy labels.
            labels.extend(y.cpu().tolist())
    # Zwracamy slownik z accuracy i F1 macro.
    return {
        # Liczymy accuracy na wszystkich probkach.
        "accuracy": accuracy_score(labels, preds),
        # Liczymy F1 macro, aby rowno traktowac klasy.
        "f1_macro": f1_score(labels, preds, average="macro"),
    }


In [4]:
# Krok 5: jawna petla treningowa (forward -> loss -> backward -> clip -> step -> zero_grad)
# Po co to robimy: uczymy model baseline CE i widzimy kazdy etap treningu w czystym PyTorch.

# Iterujemy po epokach treningu.
for epoch in range(EPOCHS):
    # Przelaczamy model w tryb treningu.
    model.train()
    # Tworzymy pasek postepu dla batchy treningowych.
    pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}")
    # Iterujemy po batchach w loaderze treningowym.
    for batch in pbar:
        # Pobieramy input_ids i przenosimy je na CPU.
        input_ids = batch["input_ids"].to(DEVICE)
        # Pobieramy attention_mask i przenosimy je na CPU.
        attention_mask = batch["attention_mask"].to(DEVICE)
        # Pobieramy etykiety i przenosimy je na CPU.
        labels = batch["label"].to(DEVICE)

        # Robimy forward pass modelu.
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        # Pobieramy logits z wyniku modelu.
        logits = outputs.logits
        # Liczymy strate CrossEntropy miedzy logits i etykietami.
        loss = criterion(logits, labels)

        # Robimy backward pass i liczymy gradienty.
        loss.backward()
        # Przycinamy gradienty dla stabilnosci uczenia.
        clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        # Aktualizujemy wagi modelu krokiem optymalizatora.
        optimizer.step()
        # Zerujemy gradienty przed kolejnym krokiem uczenia.
        optimizer.zero_grad(set_to_none=True)

        # Wyswietlamy aktualna strate na pasku postepu.
        pbar.set_postfix({"loss": float(loss.detach().cpu())})

# Liczymy metryki na zbiorze walidacyjnym po treningu.
val_metrics = evaluate(model, val_loader)
# Liczymy metryki na zbiorze testowym po treningu.
test_metrics = evaluate(model, test_loader)
# Wypisujemy metryki walidacyjne.
print("Validation:", val_metrics)
# Wypisujemy metryki testowe.
print("Test:", test_metrics)


Epoch 1/4:   0%|          | 0/485 [00:00<?, ?it/s]

Epoch 2/4:   0%|          | 0/485 [00:00<?, ?it/s]

Epoch 3/4:   0%|          | 0/485 [00:00<?, ?it/s]

Epoch 4/4:   0%|          | 0/485 [00:00<?, ?it/s]

Validation: {'accuracy': 0.7298969072164948, 'f1_macro': 0.6316952360155766}
Test: {'accuracy': 0.7298969072164948, 'f1_macro': 0.6149108484748623}


In [5]:
# Krok 6: zapis baseline modelu i metryk
# Po co to robimy: zapisujemy artefakty, aby moc je porownac z distilled i int8 w kolejnych etapach.

# Ustalamy katalog zapisu baseline modelu.
save_dir = "../artifacts/student_fp32_baseline"
# Wczytujemy tokenizer studenta do zapisu razem z modelem.
tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL_NAME)
# Zapisujemy wytrenowany model baseline na dysk.
model.save_pretrained(save_dir)
# Zapisujemy tokenizer baseline na dysk.
tokenizer.save_pretrained(save_dir)

# Otwieramy plik JSON do zapisu metryk baseline.
with open("../outputs/metrics_student_baseline.json", "w", encoding="utf-8") as f:
    # Zapisujemy metryki validation/test w czytelnym formacie.
    json.dump({"validation": val_metrics, "test": test_metrics}, f, indent=2)

# Potwierdzamy, ze zapis modelu i metryk sie udal.
print("Zapisano baseline model i metryki")


vocab.txt: 0.00B [00:00, ?B/s]

Zapisano baseline model i metryki
